In [1]:
import numpy as np
import scanpy as sc
import spapros as sp
import pandas as pd
import anndata as ad
import os

os.makedirs("outputs", exist_ok=True)

# ============================================================
# TOGGLE HARD FILTERING (True/False)
# ============================================================

USE_HARD_TECH_FILTER = True          # <---- change this to compare
TECH_THRESHOLD = 0.6                 # recommended for MERFISH-like constraints


# ============================================================
# 0. LOAD + SUBSET
# ============================================================

adata_full = ad.read_h5ad("/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/abc_atlas.h5ad", backed='r')

np.random.seed(42)
idx = np.sort(np.random.choice(adata_full.n_obs, size=200000, replace=False))
adata = adata_full[idx, :].to_memory()

sc.pp.normalize_total(adata)
sc.pp.log1p(adata)


# ============================================================
# 1. CELLTYPE DISCOVERY (Leiden)
# ============================================================

sc.tl.pca(adata, n_comps=50)
sc.pp.neighbors(adata, n_pcs=50)
sc.tl.leiden(adata, key_added="celltype")
adata.obs["celltype"] = adata.obs["celltype"].astype(str)


# ============================================================
# 2. TECHNOLOGY CONSTRAINTS: compute penalties
# ============================================================

sp.ut.get_expression_quantile(adata, q=0.99, normalise=False, log1p=False, zeros_to_nan=False)
sp.ut.get_expression_quantile(adata, q=0.9,  normalise=False, log1p=False, zeros_to_nan=True)

pen_low  = sp.ut.plateau_penalty_kernel(var=0.1, x_min=1.0)
pen_high = sp.ut.plateau_penalty_kernel(var=0.5, x_max=2.0)

adata.var["pen_low"]  = pen_low(adata.var['quantile_0.9 expr > 0'])
adata.var["pen_high"] = pen_high(adata.var['quantile_0.99'])
adata.var["tech_penalty"] = adata.var["pen_low"] * adata.var["pen_high"]

adata.var["tech_penalty"].to_csv("outputs/technology_penalties.csv")


# ============================================================
# 3. APPLY HARD OR SOFT FILTER
# ============================================================

if USE_HARD_TECH_FILTER:
    print("⚠️ Using HARD TECHNOLOGY FILTER")
    good_genes = adata.var_names[adata.var["tech_penalty"] > TECH_THRESHOLD]
    adata = adata[:, good_genes].copy()
else:
    print("ℹ️ Using SOFT TECHNOLOGY PENALTIES (no genes removed)")
    # keep all genes – penalties only reduce importance


print("Remaining genes:", adata.n_vars)

def save_panel_csv(gene_list, file_path):
    """
    Saves a list of gene IDs as a CSV file with header 'Ensembl_ID'.
    """
    df = pd.DataFrame(gene_list, columns=["Ensembl_ID"])
    df.to_csv(file_path, index=False)


/tmp/ipykernel_37503/4061298612.py:38: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, key_added="celltype")
IOStream.flush timed out
IOStream.flush timed out


⚠️ Using HARD TECHNOLOGY FILTER
Remaining genes: 26349


In [4]:

# ============================================================
# 4. FIRST SELECTOR (DE-only)
# ============================================================
adata.var["highly_variable"] = True
selector1 = sp.se.ProbesetSelector(
    adata,
    celltype_key="celltype",
    genes_key="highly_variable",   # esiste
    n=None,
    n_pca_genes=0,
    DE_penalties=["tech_penalty"],
    verbosity=1
)
selector1.select_probeset()

genes_run1 = selector1.probeset.query("selection").index.tolist()
print("Run1 (DE-only) genes:", len(genes_run1))




SPAPROS PROBESET SELECTION:                                                                      3:25:20
Train baseline forest based on DE genes................... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   4/4 3:25:14
  Select DE genes......................................... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46/46 0:00:01
  Train prior forest for DE_baseline forest............... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   3/3 0:32:51
  Iteratively add DE genes to DE_baseline forest.......... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   3/3 2:11:36
  Train final baseline forest on all celltypes............ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   3/3 0:31:58
Compile probeset list..................................... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  100% 0:00:02
FINISHED

Run1 (DE-only) genes: 209


In [ ]:
# ============================================================
# 5. SECOND SELECTOR (PCA + DE)
# ============================================================

adata2 = adata[:, [g for g in adata.var_names if g not in genes_run1]].copy()

TARGET_SIZES = [1000]

panels = {}  # store panels

for N in TARGET_SIZES:

    remaining = max(0, N - len(genes_run1))

    selector2 = sp.se.ProbesetSelector(
        adata2,
        celltype_key="celltype",
        n=remaining,
        n_pca_genes=100,
        pca_penalties=["tech_penalty"],
        DE_penalties=["tech_penalty"],
        verbosity=1
    )
    selector2.select_probeset()

    genes_run2 = selector2.probeset.query("selection").index.tolist()

    # -------------------------------
    # BUILD PANEL 1000
    # -------------------------------
    full_panel_1000 = genes_run1 + genes_run2
    panels[N] = full_panel_1000

    df1000 = pd.DataFrame({
        "gene": full_panel_1000,
        "source": (["DE_run1"] * len(genes_run1)) +
                  (["PCA+DE_run2"] * len(genes_run2)),
        "rank": np.arange(1, len(full_panel_1000) + 1)
    })

    suffix = "hard" if USE_HARD_TECH_FILTER else "soft"

    # Detailed format
    fname1000 = f"outputs/GenePanel_{N}_{suffix}.csv"
    df1000.to_csv(fname1000, index=False)

    # Simple MERFISH-like format
    fname1000_simple = f"outputs/GenePanel_{N}_{suffix}_simple.csv"
    save_panel_csv(full_panel_1000, fname1000_simple)

    print(f"Saved panels:\n  {fname1000}\n  {fname1000_simple}")


    # ============================================================
    # 6. THIRD SELECTOR — selects 500 more genes
    # ============================================================

    EXTRA = 500

    # Remove already selected genes
    genes_taken = set(full_panel_1000)
    genes_available = [g for g in adata.var_names if g not in genes_taken]

    adata3 = adata[:, genes_available].copy()

    selector3 = sp.se.ProbesetSelector(
        adata3,
        celltype_key="celltype",
        n=EXTRA,
        n_pca_genes=100,
        pca_penalties=["tech_penalty"],
        DE_penalties=["tech_penalty"],
        verbosity=1
    )
    selector3.select_probeset()

    genes_run3 = selector3.probeset.query("selection").index.tolist()

    # -------------------------------
    # BUILD PANEL 1500
    # -------------------------------
    full_panel_1500 = full_panel_1000 + genes_run3

    df1500 = pd.DataFrame({
        "gene": full_panel_1500,
        "source": (["DE_run1"] * len(genes_run1)) +
                  (["PCA+DE_run2"] * len(genes_run2)) +
                  (["EXTRA_run3"] * len(genes_run3)),
        "rank": np.arange(1, len(full_panel_1500) + 1)
    })

    fname1500 = f"outputs/GenePanel_{N+EXTRA}_{suffix}.csv"
    df1500.to_csv(fname1500, index=False)

    fname1500_simple = f"outputs/GenePanel_{N+EXTRA}_{suffix}_simple.csv"
    save_panel_csv(full_panel_1500, fname1500_simple)

    print(f"Saved panels:\n  {fname1500}\n  {fname1500_simple}")
